In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox

#Class Transaksi
class Transaksi:
    def __init__(self, tanggal, kategori, jumlah):
        self.tanggal  = tanggal
        self.kategori = kategori
        self.jumlah   = int(jumlah)

#Class Dompet
# untuk menambahkan pengeluaran, sisa saldo, dan menyimpan file 
class Dompet:
    def __init__(self, saldo_awal):
        self.saldo_awal  = saldo_awal
        self.transaksi   = []

    def tambah_pengeluaran(self, transaksi):
        self.transaksi.append(transaksi)

    def sisa_saldo(self):
        total_keluar = sum(t.jumlah for t in self.transaksi)
        return self.saldo_awal - total_keluar

    def simpan_ke_file(self):
        with open("riwayat.txt", "w") as f:
            f.write(f"Saldo Awal : Rp {self.saldo_awal:,}\n")
            f.write("-" * 40 + "\n")
            for t in self.transaksi:
                f.write(f"{t.tanggal} | {t.kategori} | Rp {t.jumlah:,}\n")
            f.write("-" * 40 + "\n")
            f.write(f"Sisa Saldo : Rp {self.sisa_saldo():,}\n")

#Class Aplikasi 
# untuk membuat tampilan GUI
class AplikasiKeuangan:
    def __init__(self, root):
        self.root   = root
        self.dompet = None

        self.root.title("Pengelola Keuangan Kost")
        self.root.geometry("500x480")
        self.root.configure(bg="#f0f0f0")

        self._build_ui()

    def _build_ui(self):
        tk.Label(self.root, text="Pengelola Keuangan Kost",
                 font=("Arial", 16, "bold"),
                 bg="#2c3e50", fg="white").pack(fill="x", pady=(0, 10))

        frame_saldo = tk.Frame(self.root, bg="#f0f0f0")
        frame_saldo.pack(pady=5)

        tk.Label(frame_saldo, text="Saldo Awal (Rp):",
                 bg="#f0f0f0", font=("Arial", 10)).grid(row=0, column=0, padx=5)
        self.entry_saldo = tk.Entry(frame_saldo, width=15, font=("Arial", 10))
        self.entry_saldo.grid(row=0, column=1, padx=5)
        tk.Button(frame_saldo, text="Simpan", bg="#27ae60", fg="white",
                  command=self._simpan_saldo).grid(row=0, column=2, padx=5)


        form = tk.LabelFrame(self.root, text="Tambah Pengeluaran",
                             bg="#f0f0f0", font=("Arial", 10, "bold"))
        form.pack(padx=15, pady=8, fill="x")

        tk.Label(form, text="Tanggal:", bg="#f0f0f0").grid(row=0, column=0, padx=8, pady=5, sticky="w")
        self.entry_tgl = tk.Entry(form, width=12)
        self.entry_tgl.grid(row=0, column=1, padx=5)

        tk.Label(form, text="Kategori:", bg="#f0f0f0").grid(row=0, column=2, padx=8)
        self.combo_kat = ttk.Combobox(form, width=14, state="readonly",
                                      values=["Makan", "Buku", "Laundry",
                                        ])
        self.combo_kat.current(0)
        self.combo_kat.grid(row=0, column=3, padx=5)

        tk.Label(form, text="Jumlah:", bg="#f0f0f0").grid(row=1, column=0, padx=8, pady=5, sticky="w")
        self.entry_jml = tk.Entry(form, width=12)
        self.entry_jml.grid(row=1, column=1, padx=5)

        tk.Button(form, text="Tambah", bg="#2980b9", fg="white", width=10,
                  command=self._tambah).grid(row=1, column=3, padx=5, pady=5)

        pb_frame = tk.Frame(self.root, bg="#f0f0f0")
        pb_frame.pack(padx=15, fill="x")

        tk.Label(pb_frame, text="Penggunaan Saldo:", bg="#f0f0f0",
                 font=("Arial", 9)).pack(anchor="w")
        self.progress = ttk.Progressbar(pb_frame, length=460, mode="determinate")
        self.progress.pack(fill="x", pady=3)
        self.lbl_status = tk.Label(pb_frame, text="", bg="#f0f0f0",
                                   font=("Arial", 9, "italic"), fg="gray")
        self.lbl_status.pack(anchor="e")

        tbl_frame = tk.LabelFrame(self.root, text="Riwayat Transaksi",
                                  bg="#f0f0f0", font=("Arial", 10, "bold"))
        tbl_frame.pack(padx=15, pady=8, fill="both", expand=True)

        self.tree = ttk.Treeview(tbl_frame,
                                 columns=("Tanggal", "Kategori", "Jumlah"),
                                 show="headings", height=7)
        for col, w in [("Tanggal", 100), ("Kategori", 150), ("Jumlah", 120)]:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=w, anchor="w")
        self.tree.pack(fill="both", expand=True)

        btn_frame = tk.Frame(self.root, bg="#f0f0f0")
        btn_frame.pack(pady=8)

        tk.Button(btn_frame, text="Hapus", width=10, bg="#e74c3c", fg="white",
                  command=self._hapus).grid(row=0, column=0, padx=5)
        tk.Button(btn_frame, text="Simpan File", width=10, bg="#8e44ad", fg="white",
                  command=self._simpan_file).grid(row=0, column=1, padx=5)
        tk.Button(btn_frame, text="Keluar", width=10, bg="gray", fg="white",
                  command=self._keluar).grid(row=0, column=2, padx=5)
#fungsi menyimpan saldo awal
    def _simpan_saldo(self):
        try:
            saldo = int(self.entry_saldo.get())
            self.dompet = Dompet(saldo)
            self._update_progress()
            messagebox.showinfo("Sukses", f"Saldo awal: Rp {saldo:,}")
        except ValueError:
            messagebox.showerror("Gagal", "Masukkan angka yang valid!")
#cek apakah sudah input saldo awal
    def _tambah(self):
        if self.dompet is None:
            messagebox.showwarning("Perhatian", "Atur saldo awal terlebih dahulu!")
            return
        try:
            tgl = self.entry_tgl.get()
            kat = self.combo_kat.get()
            jml = int(self.entry_jml.get())

            tx = Transaksi(tgl, kat, jml)
            self.dompet.tambah_pengeluaran(tx)
            self.tree.insert("", "end", values=(tgl, kat, f"Rp {jml:,}"))
            self._update_progress()

            self.entry_jml.delete(0, "end")
            messagebox.showinfo("Sukses", f"{kat} — Rp {jml:,} dicatat!")
        except ValueError:
            messagebox.showerror("Gagal", "Jumlah harus berupa angka!")
#Fungsi untuk menghapus transaksi
    def _hapus(self):
        sel = self.tree.selection()
        if not sel:
            messagebox.showerror("Gagal", "Pilih transaksi dahulu!")
            return
        idx = self.tree.index(sel[0])
        self.dompet.transaksi.pop(idx)
        self.tree.delete(sel[0])
        self._update_progress()
        messagebox.showinfo("Sukses", "Transaksi dihapus.")
#Fungsi untuk menyimpan file dalam bentuk txt
    def _simpan_file(self):
        if self.dompet is None or not self.dompet.transaksi:
            messagebox.showwarning("Kosong", "Belum ada data untuk disimpan!")
            return
        self.dompet.simpan_ke_file()
        messagebox.showinfo("Sukses", "Riwayat disimpan ke riwayat.txt")
#Fungsi untuk update progress
    def _update_progress(self):
        if self.dompet is None:
            return
        total_keluar = sum(t.jumlah for t in self.dompet.transaksi)
        pct = (total_keluar / self.dompet.saldo_awal * 100) if self.dompet.saldo_awal > 0 else 0
        self.progress["value"] = min(pct, 100)

        style = ttk.Style()
        if pct < 50:
            warna, status = "#27ae60", f"Masih Aman Nih!— Sisa Rp {self.dompet.sisa_saldo():,}"
        elif pct < 70:
            warna, status = "#f39c12", f"Mulai Hemat Sih! Sisa Rp {self.dompet.sisa_saldo():,}"
        else:
            warna, status = "#e74c3c", f"Wah Udah Sih! Sisa Rp {self.dompet.sisa_saldo():,}"

        style.configure("bar.Horizontal.TProgressbar",
                        background=warna, troughcolor="#ddd", thickness=16)
        self.progress.configure(style="bar.Horizontal.TProgressbar")
        self.lbl_status.config(text=f"{pct:.0f}%  |  {status}")
#fungsi untuk keluar program
    def _keluar(self):
        if messagebox.askyesno("Konfirmasi", "Yakin ingin keluar?"):
            self.root.destroy()
#main
if __name__ == "__main__":
    root = tk.Tk()
    app  = AplikasiKeuangan(root)
    root.mainloop()